In [0]:
%pip install matplotlib
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

source_table = "workspace.entity_resolution_project.company_er_scores"
target_table = "workspace.entity_resolution_project.company_er_decisions"

scores = spark.table(source_table)

HIGH_CONF = 0.80
LOW_FLOOR = 0.55

MID_CONF = 0.70
SMALL_GAP = 0.05

In [0]:
# Ranking + entropy per input record
group_window = Window.partitionBy("left_row_key")
rank_window = Window.partitionBy("left_row_key").orderBy(F.desc("composite_score"))

scored = (
    scores
    .withColumn("candidate_rank", F.row_number().over(rank_window))
    .withColumn("score_sum", F.sum("composite_score").over(group_window))
    .withColumn("candidate_count", F.count("*").over(group_window))
    .withColumn(
        "score_probability",
        F.when(F.col("score_sum") > 0, F.col("composite_score") / F.col("score_sum"))
         .otherwise(F.lit(0.0))
    )
    .withColumn(
        "entropy_component",
        F.when(
            F.col("score_probability") > 0,
            -F.col("score_probability") * F.log(F.col("score_probability"))
        ).otherwise(F.lit(0.0))
    )
    .withColumn("entropy", F.sum("entropy_component").over(group_window))
    .withColumn(
        "entropy_norm",
        F.when(
            F.col("candidate_count") > 1,
            F.col("entropy") / F.log(F.col("candidate_count"))
        ).otherwise(F.lit(0.0))
    )
)


In [0]:
# Top1 - top2 gap per input record
top_scores = (
    scored
    .filter(F.col("candidate_rank") <= 2)
    .groupBy("left_row_key")
    .agg(
        F.max(F.when(F.col("candidate_rank") == 1, F.col("composite_score"))).alias("top1_score"),
        F.max(F.when(F.col("candidate_rank") == 2, F.col("composite_score"))).alias("top2_score")
    )
    .withColumn(
        "score_gap_top1_top2",
        F.col("top1_score") - F.coalesce(F.col("top2_score"), F.lit(0.0))
    )
)

scored_with_gap = scored.join(top_scores, on="left_row_key", how="left")

In [0]:
decisions = (
    scored_with_gap
    .withColumn(
        "high_conf_match",
        F.col("top1_score") >= F.lit(HIGH_CONF)
    )
    .withColumn(
        "mid_conf_gap_match",
        (F.col("top1_score") >= F.lit(MID_CONF)) &
        (F.col("score_gap_top1_top2") >= F.lit(SMALL_GAP))
    )
    .withColumn(
        "is_match_combo",
        F.col("high_conf_match") | F.col("mid_conf_gap_match")
    )
    
    .withColumn(
        "decision",
        F.when(
            (F.col("candidate_rank") == 1) & F.col("is_match_combo"),
            F.lit("MATCH")
        )
        .when(
            (F.col("candidate_rank") == 1) &
            (F.col("top1_score") >= F.lit(LOW_FLOOR)),
            F.lit("AMBIGUOUS")
        )
        .otherwise(F.lit("NO_MATCH"))
    )
    .withColumn(
        "decision_method",
        F.when(F.col("decision") == "MATCH", F.lit("score_gap_combo_gate"))
         .when(F.col("decision") == "AMBIGUOUS", F.lit("llm_tiebreaker_gate"))
         .otherwise(F.lit("low_score_or_not_top_candidate"))
    )

    .withColumn(
        "decision_rule",
        F.when(
            (F.col("candidate_rank") == 1) & F.col("high_conf_match"),
            F.lit("high_conf_score")
        )
        .when(
            (F.col("candidate_rank") == 1) & F.col("mid_conf_gap_match"),
            F.lit("mid_conf_with_gap")
        )
        .when(
            (F.col("candidate_rank") == 1) &
            (F.col("top1_score") >= F.lit(LOW_FLOOR)),
            F.lit("needs_llm_review")
        )
        .when(
            (F.col("candidate_rank") == 1) &
            (F.col("top1_score") < F.lit(LOW_FLOOR)),
            F.lit("low_score")
        )
        .otherwise(F.lit("not_top_candidate"))
    )

    .withColumn(
        "decision_reason",
        F.concat_ws(
            " | ",
            F.concat(F.lit("decision_rule="), F.col("decision_rule")),
            F.concat(F.lit("rank="), F.col("candidate_rank")),
            F.concat(F.lit("score="), F.round(F.col("composite_score"), 3)),
            F.concat(F.lit("top1="), F.round(F.col("top1_score"), 3)),
            F.concat(F.lit("top2="), F.round(F.col("top2_score"), 3)),
            F.concat(F.lit("gap="), F.round(F.col("score_gap_top1_top2"), 3)),
            F.concat(F.lit("entropy_norm="), F.round(F.col("entropy_norm"), 3)),
            F.concat(F.lit("name_similarity="), F.round(F.col("name_similarity"), 3)),
            F.concat(F.lit("semantic_similarity="), F.round(F.col("semantic_similarity"), 3)),
            F.concat(F.lit("country_match="), F.col("country_match")),
            F.concat(F.lit("city_match="), F.col("city_match"))
        )
    )
)

In [0]:
(
    decisions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

In [0]:
display(
    decisions
    .filter(F.col("candidate_rank") == 1)
    .groupBy("decision", "decision_method")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    decisions
    .filter(F.col("candidate_rank") == 1)
    .groupBy("decision")
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("top1_score"), 3).alias("avg_top1"),
        F.round(F.avg("score_gap_top1_top2"), 3).alias("avg_gap"),
        F.round(F.avg("entropy_norm"), 3).alias("avg_entropy")
    )
    .orderBy("decision")
)

In [0]:
top_only = decisions.filter(F.col("candidate_rank") == 1)

total = top_only.count()
ambiguous_count = top_only.filter(F.col("decision") == "AMBIGUOUS").count()
match_count = top_only.filter(F.col("decision") == "MATCH").count()
no_match_count = top_only.filter(F.col("decision") == "NO_MATCH").count()

print("Threshold diagnostics")
print(f"Total input records: {total}")
print(f"AMBIGUOUS: {ambiguous_count} ({ambiguous_count / total:.1%})")
print(f"MATCH: {match_count} ({match_count / total:.1%})")
print(f"NO_MATCH: {no_match_count} ({no_match_count / total:.1%})")

In [0]:
display(
    decisions
    .filter(F.col("candidate_rank") == 1)
    .select(
        "decision",
        "decision_rule",
        "left_row_key",
        "right_row_key",
        "left_company_name",
        "right_company_name",
        "left_country_code",
        "right_country_code",
        "left_city",
        "right_city",
        "right_website_domain",
        F.round(F.col("top1_score"), 3).alias("top1_score"),
        F.round(F.col("top2_score"), 3).alias("top2_score"),
        F.round(F.col("score_gap_top1_top2"), 3).alias("score_gap"),
        F.round(F.col("name_similarity"), 3).alias("name_similarity"),
        F.round(F.col("semantic_similarity"), 3).alias("semantic_similarity"),
        "country_match",
        "city_match",
        F.round(F.col("entropy_norm"), 3).alias("entropy_norm"),
        "decision_reason"
    )
    .orderBy("decision", F.desc("top1_score"))
)

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt

plot_df = (
    decisions
    .filter(F.col("candidate_rank") == 1)
    .select(
        "decision",
        "top1_score",
        "score_gap_top1_top2",
        "entropy_norm"
    )
    .toPandas()
)

print("Stats by status")
print(
    plot_df
    .groupby("decision")
    .agg(
        count=("top1_score", "count"),
        avg_top1=("top1_score", "mean"),
        avg_gap=("score_gap_top1_top2", "mean"),
        avg_entropy=("entropy_norm", "mean")
    )
    .round(3)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {
    "MATCH": "#2E8B57",
    "AMBIGUOUS": "#6A5ACD",
    "NO_MATCH": "#C76E3A",
}

for status, color in colors.items():
    values = plot_df.loc[plot_df["decision"] == status, "top1_score"].dropna()
    if len(values) > 0:
        axes[0].hist(
            values,
            bins=25,
            alpha=0.65,
            label=f"{status} (n={len(values)})",
            color=color
        )

axes[0].axvline(HIGH_CONF, color="black", linestyle="--", label=f"HIGH_CONF={HIGH_CONF}")
axes[0].axvline(LOW_FLOOR, color="gray", linestyle="--", label=f"LOW_FLOOR={LOW_FLOOR}")
axes[0].set_title("Score distribution by resolution status")
axes[0].set_xlabel("Top-1 Composite Score")
axes[0].set_ylabel("Records")
axes[0].legend()

box_data = []
box_labels = []

for status in ["AMBIGUOUS", "MATCH", "NO_MATCH"]:
    values = plot_df.loc[plot_df["decision"] == status, "entropy_norm"].dropna()
    if len(values) > 0:
        box_data.append(values)
        box_labels.append(status)

axes[1].boxplot(box_data, labels=box_labels)
axes[1].set_title("Entropy by resolution status")
axes[1].set_ylabel("Normalised Shannon entropy")
axes[1].grid(True, axis="y", alpha=0.4)

plt.tight_layout()
plt.show()